# autofragment: mmCIF (bio) example

This notebook runs the **bio** workflow on an mmCIF file and writes a JSON fragmentation output.

Input: `examples/4ZLY.cif`
Output: `examples/output/4ZLY.fragments.json`

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd()
examples_dir = repo_root / "examples"
input_cif = examples_dir / "4ZLY.cif"
output_dir = examples_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
output_json = output_dir / "4ZLY.fragments.json"

input_cif, output_json

In [ ]:
# Run the CLI subcommand: autofragment bio
# (This expects the `autofragment` console script to be available in the current environment.)
cmd = [
    "autofragment",
    "bio",
    "--input",
    str(input_cif),
    "--output",
    str(output_json),
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

output_json

In [ ]:
# Load the output and inspect the flat fragments list
data = json.loads(output_json.read_text())

version = data.get("version")
n_frags = len(data.get("chemical_system", {}).get("fragments", []))
partitioning = data.get("partitioning", {})

print("version:", version)
print("partitioning:", partitioning)
print("n_fragments_written:", n_frags)

# Example: mmCIF bio system (flat residue fragments)

This notebook shows common `autofragment` workflows for **mmCIF biological systems** using the provided file `examples/4ZLY.cif`.

Topics:
- Partition an mmCIF file into a **flat list** of residue-level fragments
- Summarize fragment IDs by prefix (chain / polymer segment / water cluster / ligand)
- Write JSON output

Requirements:
- `gemmi` must be available (install via `pip install autofragment[bio]`).

In [ ]:
from pathlib import Path

import autofragment as af

repo_root = Path.cwd()
examples_dir = repo_root / 'examples' if (repo_root / 'examples').exists() else repo_root
cif_path = examples_dir / '4ZLY.cif'

print('autofragment version:', af.__version__)
print('mmCIF file exists:', cif_path.exists(), '-', cif_path)

## 1) Create a BioPartitioner

The `BioPartitioner` lives in `autofragment.partitioners.bio` and requires the optional `bio` extra.

In [ ]:
try:
    from autofragment.partitioners.bio import BioPartitioner
except ImportError as e:
    raise ImportError(
        'Bio support is not installed. Run: pip install autofragment[bio]'
    ) from e

bp = BioPartitioner(
    # water_clusters=None => auto-detect per chain (roughly sqrt(n_waters))
    water_clusters=None,
    water_cluster_method='kmeans_constrained',
    infer_bonds=True,
)

bp

## 2) Partition the mmCIF file

The result is a `FragmentTree` containing a flat `fragments` list and optional inferred `interfragment_bonds`.

In [ ]:
try:
    tree = bp.partition_file(str(cif_path))
except ImportError as e:
    # This can happen if the constrained clustering extra isn't installed
    print('Optional dependency missing for kmeans_constrained; retrying with kmeans.')
    print('Details:', e)
    bp = BioPartitioner(water_clusters=None, water_cluster_method='kmeans', infer_bonds=True)
    tree = bp.partition_file(str(cif_path))

print('fragments:', len(tree.fragments))
print('interfragment_bonds:', len(tree.interfragment_bonds))
print('partitioning:', tree.partitioning)

## 3) Summarize fragment IDs by prefix

Bio fragment IDs are stable and readable. They look like:

- `CHAIN_A_HELIX|A:GLY:389` (polymer residue in a HELIX segment)
- `CHAIN_A_WCL2|A:HOH:601` (water residue in water-cluster 2)
- `CHAIN_A_LIG|A:IMD:701` (ligand residue)

We can group by the prefix before the `|`.

In [ ]:
from collections import Counter

def prefix(fragment_id: str) -> str:
    return fragment_id.split('|', 1)[0] if '|' in fragment_id else fragment_id

counts = Counter(prefix(f.id) for f in tree.fragments)
for k, v in counts.most_common(25):
    print(f'{k}: {v}')

# Show a few example IDs
for f in tree.fragments[:10]:
    print(f.id)

## 4) Write JSON output

The output JSON contains a flat fragments list under `fragments` (no nested fragment trees).

In [ ]:
out_path = examples_dir / '4ZLY_partitioned.json'
tree.to_json(str(out_path))
print('wrote:', out_path)
print('bytes:', out_path.stat().st_size)

## CLI equivalent

```bash
autofragment bio --input examples/4ZLY.cif --output examples/4ZLY_partitioned.json
```

Optional flags:
- `--water-clusters N` to fix the number of water clusters
- `--water-cluster-method kmeans|...` to change the clustering method
- `--no-infer-bonds` to disable inferred interfragment bonds